In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib agg

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "/Users/dustinm/projects/research/ma-lab/SpatialAgent/src")
DATA_DIR = Path.cwd()

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores.chroma import Chroma

documentation_dirpath = Path(
    "/Users/dustinm/projects/research/ma-lab/SpatialAgent/src/agents/data_analysis_agent/rag_impl/documentation/squidpy/"
)

function_names = [
    "squidpy.gr.centrality_scores.md",
    "squidpy.gr.co_occurrence.md",
    "squidpy.gr.interaction_matrix.md",
    "squidpy.gr.ligrec.md",
    "squidpy.gr.mask_graph.md",
    "squidpy.gr.nhood_enrichment.md",
    "squidpy.gr.ripley.md",
    "squidpy.gr.sepal.md",
    "squidpy.gr.spatial_autocorr.md",
    "squidpy.gr.spatial_neighbors.md",
    "squidpy.im.calculate_image_features.md",
    "squidpy.im.process.md",
    "squidpy.im.segment.md",
    "squidpy.pl.centrality_scores.md",
    "squidpy.pl.co_occurrence.md",
    "squidpy.pl.extract.md",
    "squidpy.pl.interaction_matrix.md",
    "squidpy.pl.ligrec.md",
    "squidpy.pl.nhood_enrichment.md",
    "squidpy.pl.ripley.md",
    "squidpy.pl.spatial_scatter.md",
    "squidpy.pl.spatial_segment.md",
    "squidpy.pl.var_by_distance.md",
]

embedder = OpenAIEmbeddings(model="text-embedding-3-small")

chroma_dirpath = DATA_DIR / "chroma"

In [ ]:
embed_values = []
for function_name in function_names:
    documentation_filepath = documentation_dirpath / function_name
    with documentation_filepath.open("r") as documentation_file:
        documentation = documentation_file.read()
    embed_values.append(documentation)

print(embed_values[0][:500])

In [ ]:
import re

embed_values = list(map(lambda text: re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text), embed_values))
print(embed_values[0][:500])

In [ ]:
import shutil

if chroma_dirpath.exists():
    shutil.rmtree(chroma_dirpath)
chroma_dirpath.mkdir()

db = Chroma.from_texts(embed_values, embedder, persist_directory=str(chroma_dirpath))

In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message=r"Relevance scores must be between 0 and 1.*",
)

query_text = "plot spatial co-occurence by cluster"
relevance_th = 0.12
results = db.similarity_search_with_relevance_scores(query_text, k=5, score_threshold=relevance_th)

In [ ]:
for result in results:
    document, score = result
    print(score, "\n", document.page_content[:500])
    print("-" * 150)